# AMTTP GPU Fraud Pipeline (RAPIDS + TabNet + LightGBM + Optional GNN/Transformer)

This notebook implements a fully GPU-accelerated, memory-safe multi‑dataset fraud detection stack with:
- RAPIDS (cuDF / cuML / cuPy) for preprocessing
- TabNet + LightGBM + (optional) GraphSAGE + (optional) Tiny Transformer
- Logistic regression blender, cost-based thresholding
- SHAP (optional) & feature importances
- Cryptographic keccak256 + ECDSA signing of a representative risk payload
- Colab-friendly resource guards (LIGHT_MODE, streaming, memory fraction cap)

Sections mirror the engineering design for transparency and reproducibility.

In [ ]:
# 1. Environment Setup (RAPIDS + Dependencies)
# NOTE: RAPIDS version must match the CUDA version provided by Colab (check nvidia-smi first).
# If RAPIDS install fails (version mismatch), fallback to CPU pandas path is automatic.

import os, sys, subprocess, textwrap, importlib

REQUIRED_PY_PKGS = [
    'lightgbm==4.5.0',
    'scikit-learn==1.5.2',
    'pytorch-tabnet==4.1.0',
    'eth-keys', 'eth-utils', 'joblib', 'shap'
]

# Torch / PyG handled separately for GPU compatibility.

def pip_install(pkgs):
    for p in pkgs:
        try:
            print(f"Installing {p} ...")
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
        except Exception as e:
            print(f"Failed to install {p}: {e}")

print("Installing core Python packages ...")
pip_install(REQUIRED_PY_PKGS)

# Attempt torch install (Colab usually preinstalls a GPU-enabled torch)
try:
    import torch
    print("Torch version:", torch.__version__)
except Exception:
    pip_install(['torch'])
    import torch

# Optional: torch-geometric
TRY_PYG = True
if TRY_PYG:
    try:
        # Basic attempt; detailed wheels sometimes needed per CUDA version
        pip_install(['torch-geometric'])
        import torch_geometric  # noqa
        print('PyG available.')
    except Exception as e:
        print('PyG install skipped / failed:', e)

# RAPIDS (optional) -- comment out if mismatch; adjust cudf-cu12 for CUDA12.
RAPIDS_SPEC = 'cudf-cu12 dask-cudf-cu12 cuml-cu12 cupy-cuda12x'
EXTRA_INDEX = 'https://pypi.nvidia.com'
INSTALL_RAPIDS = True
if INSTALL_RAPIDS:
    try:
        print('Attempting RAPIDS install...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *RAPIDS_SPEC.split(), '--extra-index-url', EXTRA_INDEX])
        import cudf, cuml, cupy  # noqa
        print('RAPIDS installed successfully.')
    except Exception as e:
        print('RAPIDS install failed; will fall back to CPU path if needed:', e)

print("Environment setup cell complete.")

In [ ]:
# 2. Verify GPU & RAPIDS Versions
import importlib, json, math

cuda_available = False
try:
    import torch
    cuda_available = torch.cuda.is_available()
except Exception:
    pass

gpu_name = None
if cuda_available:
    try:
        gpu_name = torch.cuda.get_device_name(0)
    except Exception:
        gpu_name = 'Unknown GPU'

rapids_info = {}
for pkg in ['cudf','cuml','cupy']:
    try:
        m = importlib.import_module(pkg)
        rapids_info[pkg] = getattr(m, '__version__', 'unknown')
    except Exception:
        rapids_info[pkg] = None

print('CUDA Available:', cuda_available)
print('GPU Name:', gpu_name)
print('RAPIDS Components:', rapids_info)
if not cuda_available or rapids_info.get('cudf') is None:
    print('⚠️ Running in CPU fallback mode for preprocessing unless GPU libs load later.')
else:
    print('✅ GPU & RAPIDS stack detected.')

In [ ]:
# 3. Configuration & Hyperparameters (Environment Driven)
import os, random, numpy as np

# Core directories (adjust for your environment / Drive mount)
DATA_DIR = os.environ.get('AMTTP_DATA_DIR', '/content/drive/MyDrive/dataehancedfraud')
OUT_DIR  = os.environ.get('AMTTP_OUT_DIR', '/content/drive/MyDrive/AMTTP_Models_GPU')
os.makedirs(OUT_DIR, exist_ok=True)

# Dataset configurations (enable/disable individually)
DATASETS = [
    {"name":"creditcard",        "path": f"{DATA_DIR}/transaction_dataset.csv",                        "enabled": True},
    {"name":"paysim",            "path": f"{DATA_DIR}/PS_20174392719_1491204439457_log.csv",           "enabled": True},
    {"name":"ethereum",          "path": f"{DATA_DIR}/Cryptocurrency_Scam_Dataset_for_DQN_Models.csv", "enabled": True},
    {"name":"ieee_transactions", "path": f"{DATA_DIR}/train_transaction.csv",                          "enabled": True},
]

# Targets & balancing
TARGET_COL = 'isFraud'
MAX_ROWS_PER_DATASET = int(os.environ.get('AMTTP_MAX_ROWS', 300_000))
MAJORITY_MULTIPLIER  = int(os.environ.get('AMTTP_MAJORITY_MULT', 3))
GLOBAL_ROW_CAP       = int(os.environ.get('AMTTP_GLOBAL_ROW_CAP', 1_200_000))

# Streaming / memory controls
STREAMING_ENABLED       = bool(int(os.environ.get('AMTTP_STREAMING', 0)))
STREAM_CHUNK_SIZE       = int(os.environ.get('AMTTP_STREAM_CHUNK', 200_000))
GPU_STREAMING_ENABLED   = bool(int(os.environ.get('AMTTP_GPU_STREAMING', 0)))
GPU_STREAM_CHUNK        = int(os.environ.get('AMTTP_GPU_STREAM_CHUNK', 500_000))
MEMORY_FRACTION_BUDGET  = float(os.environ.get('AMTTP_MAX_MEMORY_FRACTION', 0.55))

# Model hyperparameters / modes
LIGHT_MODE         = bool(int(os.environ.get('AMTTP_LIGHT_MODE', 0)))
TABNET_EPOCHS      = int(os.environ.get('AMTTP_TABNET_EPOCHS', 20))
TABNET_BATCH_SIZE  = int(os.environ.get('AMTTP_TABNET_BATCH', 4096))
TABNET_VBATCH_SIZE = int(os.environ.get('AMTTP_TABNET_VBATCH', 1024))
LGBM_ESTIMATORS    = int(os.environ.get('AMTTP_LGBM_ESTIMATORS', 1000))
SEQ_WINDOW         = int(os.environ.get('AMTTP_SEQ_WINDOW', 20))
MAX_GRAPH_EDGES    = int(os.environ.get('AMTTP_MAX_GRAPH_EDGES', 300_000))
MAX_SHAP_SAMPLES   = int(os.environ.get('AMTTP_SHAP_SAMPLES', 5000))
FALSE_POSITIVE_COST= float(os.environ.get('AMTTP_COST_FP', 1.0))
FALSE_NEGATIVE_COST= float(os.environ.get('AMTTP_COST_FN', 5.0))
FORCE_GPU          = bool(int(os.environ.get('AMTTP_FORCE_GPU', 0)))

# Deterministic seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"LIGHT_MODE={LIGHT_MODE} STREAMING_ENABLED={STREAMING_ENABLED} GPU_STREAMING_ENABLED={GPU_STREAMING_ENABLED} FORCE_GPU={FORCE_GPU}")
print(f"Row caps: per-dataset={MAX_ROWS_PER_DATASET} global={GLOBAL_ROW_CAP} memory_fraction={MEMORY_FRACTION_BUDGET}")

In [ ]:
# 4. Utility: Deterministic Seeding & Memory Cleanup
import gc
try:
    import cupy as cp
except Exception:
    cp = None

try:
    import psutil
    PSUTIL_AVAILABLE = True
except Exception:
    PSUTIL_AVAILABLE = False

import torch
if torch.cuda.is_available():
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

def cleanup_memory():
    gc.collect()
    if cp is not None:
        try:
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
        except Exception:
            pass

def get_memory_usage():
    if PSUTIL_AVAILABLE:
        vm = psutil.virtual_memory()
        return {"total": vm.total, "used": vm.used, "percent": vm.percent}
    return None

print("Utility functions ready. Memory snapshot:", get_memory_usage())

In [ ]:
# 5. GPU Data Processing Helpers (Label Inference, Encoding, Scaling)
import pandas as pd

GPU_AVAILABLE = False
try:
    import cudf  # type: ignore
    from cuml.preprocessing import StandardScaler as cuStandardScaler  # type: ignore
    GPU_AVAILABLE = True
except Exception:
    cudf = None  # type: ignore
    cuStandardScaler = None  # type: ignore

LABEL_CANDIDATES = {
    'creditcard': ['FLAG','Class','class'],
    'paysim': ['isFraud','is_fraud','fraud'],
    'ethereum': ['Is_Scam','is_scam','label','isFraud'],
    'ieee_transactions': ['isFraud','is_fraud']
}
GENERIC_LABELS = ['isfraud','fraud','flag','label','class','target','y']

def infer_label_column_gdf(gdf, dataset_name: str, target_col: str = TARGET_COL):
    if target_col in gdf.columns:
        return gdf
    lower = {c.lower(): c for c in gdf.columns}
    for c in LABEL_CANDIDATES.get(dataset_name, []) + GENERIC_LABELS:
        if c.lower() in lower:
            return gdf.rename(columns={lower[c.lower()]: target_col})
    return gdf

def binaryize_target_gdf(gdf, target_col: str = TARGET_COL):
    import cudf
    if not cudf.api.types.is_numeric_dtype(gdf[target_col]):
        codes = gdf[target_col].astype('category').cat.codes
        gdf[target_col] = (codes == codes.max()).astype('int8')
    else:
        u = gdf[target_col].dropna().unique().to_cupy()
        if len(u) == 2:
            mx = int(u.max())
            gdf[target_col] = (gdf[target_col] == mx).astype('int8')
        else:
            gdf[target_col] = (gdf[target_col] > 0).astype('int8')
    return gdf

def cap_and_encode_categoricals(gdf, cat_cols):
    """Cap high-cardinality categories (top 50) & integer encode via merge (memory safe)."""
    import cudf
    for c in cat_cols:
        top = gdf[c].value_counts().head(50).index
        gdf[c] = gdf[c].where(gdf[c].isin(top), 'other')
        gdf[c] = gdf[c].astype('str')
        uniq = gdf[c].unique().to_pandas().tolist()
        mapping = {v:i for i,v in enumerate(uniq, start=1)}
        mp = pd.DataFrame(list(mapping.items()), columns=[c, f"{c}__id"])
        mp_g = cudf.from_pandas(mp)
        gdf = gdf.merge(mp_g, on=c, how='left').drop(columns=[c]).rename(columns={f"{c}__id": c})
        gdf[c] = gdf[c].fillna(0).astype('int32')
    return gdf

def scale_numeric_columns(gdf, num_cols):
    if not num_cols or cuStandardScaler is None:
        return gdf
    scaler = cuStandardScaler()
    gdf[num_cols] = scaler.fit_transform(gdf[num_cols])
    return gdf

print("GPU helper functions loaded. GPU_AVAILABLE=", GPU_AVAILABLE)

In [ ]:
# 6. GPUMemoryEfficientDataProcessor Class (Full & Streaming)
from typing import List, Dict, Optional

class GPUMemoryEfficientDataProcessor:
    """Sequential GPU (or fallback CPU) dataset processor with memory safety.
    - Full GPU or chunked GPU streaming (cuDF) when GPU_AVAILABLE.
    - CPU fallback (pandas) + optional streaming when GPU unavailable.
    - Balancing + per-dataset row cap applied.
    - Frees memory after each dataset.
    """
    def __init__(self,
                 datasets: List[Dict],
                 target_col: str = TARGET_COL,
                 gpu_streaming: bool = False,
                 gpu_stream_chunk: int = 500_000,
                 cpu_streaming: bool = False,
                 cpu_stream_chunk: int = 200_000,
                 force_gpu: bool = False):
        self.datasets = datasets
        self.target_col = target_col
        self.gpu_streaming = gpu_streaming
        self.gpu_stream_chunk = gpu_stream_chunk
        self.cpu_streaming = cpu_streaming
        self.cpu_stream_chunk = cpu_stream_chunk
        self.force_gpu = force_gpu

    # ---- CPU Helpers ----
    def _binaryize_cpu(self, df: pd.DataFrame):
        if not pd.api.types.is_numeric_dtype(df[self.target_col]):
            codes = df[self.target_col].astype('category').cat.codes
            df[self.target_col] = (codes == codes.max()).astype('int8')
        else:
            u = df[self.target_col].dropna().unique()
            if len(u) == 2:
                mx = max(u)
                df[self.target_col] = (df[self.target_col] == mx).astype('int8')
            else:
                df[self.target_col] = (df[self.target_col] > 0).astype('int8')
        return df

    def _process_cpu_full(self, path: str, name: str) -> Optional[str]:
        if not os.path.exists(path):
            print(f"  ❌ Missing file: {path}")
            return None
        df = pd.read_csv(path)
        df = df.rename(columns={c: self.target_col for c in df.columns if c.lower() == self.target_col.lower()})
        if self.target_col not in df.columns:
            print(f"  ⚠️ No target in {name}; skipping.")
            return None
        df = df.dropna(thresh=int(len(df.columns)*0.5))
        num_cols = [c for c in df.columns if c != self.target_col and pd.api.types.is_numeric_dtype(df[c])]
        cat_cols = [c for c in df.columns if c not in num_cols and c != self.target_col]
        if num_cols:
            med = df[num_cols].median(); df[num_cols] = df[num_cols].fillna(med)
        for c in cat_cols:
            mv = df[c].mode(); fill = mv.iloc[0] if len(mv) else 'unknown'; df[c] = df[c].fillna(fill)
            vc = df[c].value_counts(); top = vc.head(50).index; df[c] = df[c].where(df[c].isin(top),'other')
            mapping = {v:i+1 for i,v in enumerate(df[c].astype(str).unique())}
            df[c] = df[c].astype(str).map(mapping).fillna(0).astype('int32')
        if num_cols:
            from sklearn.preprocessing import StandardScaler
            sc = StandardScaler(); df[num_cols] = sc.fit_transform(df[num_cols])
        df = self._binaryize_cpu(df)
        out = os.path.join(OUT_DIR, f"{name}_processed.parquet")
        try:
            df.to_parquet(out, index=False, compression='snappy')
        except Exception:
            alt = out.replace('.parquet','.csv.gz'); df.to_csv(alt, index=False, compression='gzip'); out = alt
        print(f"  💾 CPU saved: {out} rows={len(df):,}")
        return out

    def _process_cpu_stream(self, path: str, name: str) -> Optional[str]:
        if not os.path.exists(path):
            print(f"  ❌ Missing file: {path}")
            return None
        cat_maps: Dict[str, Dict[str,int]] = {}
        collected: List[pd.DataFrame] = []
        total_fraud = 0; total_legit = 0
        for chunk in pd.read_csv(path, chunksize=self.cpu_stream_chunk):
            if self.target_col not in chunk.columns:
                # try to infer
                lower = {c.lower(): c for c in chunk.columns}
                if self.target_col.lower() in lower:
                    chunk = chunk.rename(columns={lower[self.target_col.lower()]: self.target_col})
                else:
                    continue
            chunk = chunk.dropna(thresh=int(len(chunk.columns)*0.5))
            if chunk.empty:
                continue
            num_cols = [c for c in chunk.columns if c != self.target_col and pd.api.types.is_numeric_dtype(chunk[c])]
            cat_cols = [c for c in chunk.columns if c not in num_cols and c != self.target_col]
            if num_cols:
                med = chunk[num_cols].median(); chunk[num_cols] = chunk[num_cols].fillna(med)
            for c in cat_cols:
                mv = chunk[c].mode(); fill = mv.iloc[0] if len(mv) else 'unknown'; chunk[c] = chunk[c].fillna(fill)
                vc = chunk[c].value_counts(); top = vc.head(50).index; chunk[c] = chunk[c].where(chunk[c].isin(top),'other')
                if c not in cat_maps: cat_maps[c] = {}
                mapping = cat_maps[c]
                for v in chunk[c].astype(str).unique():
                    if v not in mapping:
                        mapping[v] = len(mapping)+1
                chunk[c] = chunk[c].astype(str).map(mapping).fillna(0).astype('int32')
            if not pd.api.types.is_integer_dtype(chunk[self.target_col]):
                chunk = self._binaryize_cpu(chunk)
            fraud = chunk[chunk[self.target_col]==1]
            legit = chunk[chunk[self.target_col]==0]
            max_fraud_allowed = MAX_ROWS_PER_DATASET // (MAJORITY_MULTIPLIER + 1)
            max_legit_allowed = MAJORITY_MULTIPLIER * max_fraud_allowed
            rem_fraud = max(0, max_fraud_allowed - total_fraud)
            rem_legit = max(0, max_legit_allowed - total_legit)
            if rem_fraud<=0 and rem_legit<=0:
                break
            take_f = min(len(fraud), rem_fraud) if rem_fraud>0 else 0
            take_l = min(len(legit), rem_legit) if rem_legit>0 else 0
            keep_parts = []
            if take_f: keep_parts.append(fraud.sample(n=take_f, random_state=SEED) if take_f < len(fraud) else fraud)
            if take_l: keep_parts.append(legit.sample(n=take_l, random_state=SEED) if take_l < len(legit) else legit)
            if not keep_parts: break
            part = pd.concat(keep_parts, ignore_index=True)
            total_fraud += (part[self.target_col]==1).sum()
            total_legit += (part[self.target_col]==0).sum()
            collected.append(part)
            if (total_fraud+total_legit) >= MAX_ROWS_PER_DATASET:
                break
        if not collected:
            print("  ⚠️ CPU streaming produced no data.")
            return None
        df = pd.concat(collected, ignore_index=True)
        num_cols_final = [c for c in df.columns if c != self.target_col and pd.api.types.is_float_dtype(df[c])]
        if num_cols_final:
            from sklearn.preprocessing import StandardScaler
            sc = StandardScaler(); df[num_cols_final] = sc.fit_transform(df[num_cols_final])
        out = os.path.join(OUT_DIR, f"{name}_processed.parquet")
        try:
            df.to_parquet(out, index=False, compression='snappy')
        except Exception:
            alt = out.replace('.parquet','.csv.gz'); df.to_csv(alt, index=False, compression='gzip'); out = alt
        print(f"  💾 CPU streaming saved: {out} rows={len(df):,}")
        return out

    # ---- GPU paths ----
    def _process_gpu_full(self, path: str, name: str) -> Optional[str]:
        if not GPU_AVAILABLE or cudf is None:
            return None
        if not os.path.exists(path):
            print(f"  ❌ Missing file: {path}"); return None
        gdf = cudf.read_csv(path)
        gdf = infer_label_column_gdf(gdf, name, TARGET_COL)
        if TARGET_COL not in gdf.columns:
            print(f"  ⚠️ No target in {name}; skipping.")
            return None
        gdf = gdf.dropna(thresh=int(len(gdf.columns)*0.5))
        num_cols = [c for c in gdf.columns if c != TARGET_COL and str(gdf[c].dtype) not in ('object','str')]
        cat_cols = [c for c in gdf.columns if c not in num_cols and c != TARGET_COL]
        if num_cols:
            med = gdf[num_cols].quantile(0.5); gdf[num_cols] = gdf[num_cols].fillna(med)
        for c in cat_cols:
            vals = gdf[c].dropna(); fill_val = vals.value_counts().index[0] if len(vals) else 'unknown'; gdf[c] = gdf[c].fillna(fill_val)
        gdf = cap_and_encode_categoricals(gdf, cat_cols)
        gdf = scale_numeric_columns(gdf, num_cols)
        gdf = binaryize_target_gdf(gdf, TARGET_COL)
        out = os.path.join(OUT_DIR, f"{name}_processed.parquet")
        gdf.to_parquet(out, compression='snappy')
        print(f"  💾 GPU saved: {out} rows={len(gdf):,}")
        cleanup_memory()
        return out

    def _process_gpu_stream(self, path: str, name: str) -> Optional[str]:
        if not GPU_AVAILABLE or cudf is None:
            return None
        if not os.path.exists(path):
            print(f"  ❌ Missing file: {path}"); return None
        try:
            iterator = cudf.read_csv(path, chunksize=self.gpu_stream_chunk)
        except Exception as e:
            print('  ⚠️ GPU chunk read failed, fallback full:', e)
            return self._process_gpu_full(path, name)
        cat_maps: Dict[str, Dict[str,int]] = {}
        chunks = []
        total_f = total_l = 0
        cap_rows = MAX_ROWS_PER_DATASET
        for gchunk in iterator:
            gchunk = infer_label_column_gdf(gchunk, name, TARGET_COL)
            if TARGET_COL not in gchunk.columns:
                continue
            gchunk = gchunk.dropna(thresh=int(len(gchunk.columns)*0.5))
            if len(gchunk)==0: continue
            num_cols = [c for c in gchunk.columns if c != TARGET_COL and str(gchunk[c].dtype) not in ('object','str')]
            cat_cols = [c for c in gchunk.columns if c not in num_cols and c != TARGET_COL]
            if num_cols:
                med = gchunk[num_cols].quantile(0.5); gchunk[num_cols] = gchunk[num_cols].fillna(med)
            # categorical capping + persistent mapping
            import cudf
            for c in cat_cols:
                top = gchunk[c].value_counts().head(50).index
                gchunk[c] = gchunk[c].where(gchunk[c].isin(top),'other').astype('str')
                if c not in cat_maps: cat_maps[c] = {}
                mapping = cat_maps[c]
                new_vals = [v for v in gchunk[c].unique().to_pandas().tolist() if v not in mapping]
                if new_vals:
                    start_id = len(mapping)+1
                    for i,v in enumerate(new_vals):
                        mapping[v] = start_id + i
                mp = pd.DataFrame(list(mapping.items()), columns=[c, f"{c}__id"])
                mp_g = cudf.from_pandas(mp)
                gchunk = gchunk.merge(mp_g, on=c, how='left').drop(columns=[c]).rename(columns={f"{c}__id": c})
                gchunk[c] = gchunk[c].fillna(0).astype('int32')
            gchunk = binaryize_target_gdf(gchunk, TARGET_COL)
            fraud = gchunk[gchunk[TARGET_COL]==1]
            legit = gchunk[gchunk[TARGET_COL]==0]
            max_f_allowed = cap_rows // (MAJORITY_MULTIPLIER + 1)
            max_l_allowed = MAJORITY_MULTIPLIER * max_f_allowed
            rem_f = max(0, max_f_allowed - total_f)
            rem_l = max(0, max_l_allowed - total_l)
            if rem_f<=0 and rem_l<=0: break
            take_f = int(min(len(fraud), rem_f)) if rem_f>0 else 0
            take_l = int(min(len(legit), rem_l)) if rem_l>0 else 0
            keep = []
            if take_f:
                keep.append(fraud.sample(n=take_f) if take_f < len(fraud) else fraud)
            if take_l:
                keep.append(legit.sample(n=take_l) if take_l < len(legit) else legit)
            if not keep: break
            part = cudf.concat(keep)
            total_f += int(part[TARGET_COL].sum())
            total_l += int(len(part) - part[TARGET_COL].sum())
            chunks.append(part)
            if (total_f+total_l) >= cap_rows: break
        if not chunks:
            print('  ⚠️ No GPU streaming data collected.')
            return None
        final_gdf = cudf.concat(chunks)
        num_cols_final = [c for c in final_gdf.columns if c != TARGET_COL and str(final_gdf[c].dtype) not in ('object','str')]
        if num_cols_final and cuStandardScaler is not None:
            scaler = cuStandardScaler(); final_gdf[num_cols_final] = scaler.fit_transform(final_gdf[num_cols_final])
        out = os.path.join(OUT_DIR, f"{name}_processed.parquet")
        final_gdf.to_parquet(out, compression='snappy')
        print(f"  💾 GPU streaming saved: {out} rows={len(final_gdf):,}")
        cleanup_memory()
        return out

    def process_all(self) -> List[str]:
        outputs: List[str] = []
        for cfg in self.datasets:
            if not cfg.get('enabled', True):
                continue
            name, path = cfg['name'], cfg['path']
            print(f"\n=== Dataset: {name} ===")
            out = None
            if GPU_AVAILABLE and (self.force_gpu or (self.gpu_streaming and not self.cpu_streaming)):
                if self.gpu_streaming:
                    out = self._process_gpu_stream(path, name)
                else:
                    out = self._process_gpu_full(path, name)
            elif GPU_AVAILABLE and self.force_gpu:
                out = self._process_gpu_full(path, name)
            else:
                if self.cpu_streaming:
                    out = self._process_cpu_stream(path, name)
                else:
                    out = self._process_cpu_full(path, name)
            if out:
                outputs.append(out)
            cleanup_memory()
        return outputs

print("GPUMemoryEfficientDataProcessor class defined.")

### 7. GPU-Accelerated Data Processor Class

This class, `GPUMemoryEfficientDataProcessor`, is the core of our RAPIDS-based preprocessing pipeline. It encapsulates all the logic for handling data on the GPU, including loading, cleaning, feature engineering, and scaling. It's designed to be memory-efficient by operating on a single dataset at a time. This directly implements the principles outlined in the migration to a GPU-native workflow.

In [ ]:

class GPUMemoryEfficientDataProcessor:
    """
    Encapsulates the GPU-native (cuDF) data preprocessing logic.
    This class directly implements the principles from the GPU migration plan:
    - Uses cuDF for all operations.
    - Handles implicit conversion errors by explicitly managing data types.
    - Provides a clear, reusable component for sequential processing.
    """
    def __init__(self, target_col: str, out_dir: str):
        if not GPU_AVAILABLE:
            raise RuntimeError("GPU (RAPIDS) not available. This processor requires cuDF and cuML.")
        self.target_col = target_col
        self.out_dir = out_dir
        os.makedirs(self.out_dir, exist_ok=True)

    def _infer_label_column(self, gdf, dataset_name: str):
        if self.target_col in gdf.columns:
            return gdf
        # Standardized mapping for known datasets
        mapping = {
            'creditcard': ['FLAG', 'Class', 'class'],
            'paysim': ['isFraud', 'is_fraud', 'fraud'],
            'ethereum': ['Is_Scam', 'is_scam', 'label', 'isFraud'],
            'ieee_transactions': ['isFraud', 'is_fraud'],
        }
        candidates = mapping.get(dataset_name, [])
        generic = ['isfraud', 'fraud', 'flag', 'label', 'class', 'target', 'y']
        lower_cols = {c.lower(): c for c in gdf.columns}

        for c in candidates + generic:
            if c.lower() in lower_cols:
                print(f"  - Found label column '{lower_cols[c.lower()]}', renaming to '{self.target_col}'")
                return gdf.rename(columns={lower_cols[c.lower()]: self.target_col})
        return gdf

    def _binaryize_target(self, gdf):
        """Ensures the target column is binary (0 or 1)."""
        if not cudf.api.types.is_numeric_dtype(gdf[self.target_col]):
            # Handle categorical targets like 'Yes'/'No'
            codes = gdf[self.target_col].astype('category').cat.codes
            gdf[self.target_col] = (codes == codes.max()).astype('int8')
        else:
            # Handle numeric targets like 0/1 or floats
            unique_vals = gdf[self.target_col].dropna().unique()
            if len(unique_vals) == 2:
                # Force to 0/1 based on the max value
                max_val = unique_vals.max()
                gdf[self.target_col] = (gdf[self.target_col] == max_val).astype('int8')
            else:
                # Assume anything > 0 is the positive class
                gdf[self.target_col] = (gdf[self.target_col] > 0).astype('int8')
        return gdf

    def process_dataset(self, csv_path: str, dataset_name: str) -> Optional[str]:
        """
        Main processing function for a single dataset. Loads, cleans, transforms,
        and saves the data using only GPU operations.
        """
        print(f"\\n🔄 GPU preprocessing: {dataset_name}")
        if not os.path.exists(csv_path):
            print(f"  ❌ Missing file: {csv_path}")
            return None

        try:
            # 1. Load directly to GPU
            gdf = cudf.read_csv(csv_path)
            gdf = self._infer_label_column(gdf, dataset_name)

            if self.target_col not in gdf.columns:
                print(f"  ❌ No label column found for {dataset_name}, skipping.")
                cleanup_memory()
                return None

            # 2. Clean and impute
            gdf = gdf.dropna(thresh=int(len(gdf.columns) * 0.5)) # Drop rows with >50% missing
            num_cols = [c for c in gdf.columns if c != self.target_col and cudf.api.types.is_numeric_dtype(gdf[c])]
            cat_cols = [c for c in gdf.columns if c not in num_cols and c != self.target_col]

            if num_cols:
                median_vals = gdf[num_cols].quantile(0.5)
                gdf[num_cols] = gdf[num_cols].fillna(median_vals)
            for c in cat_cols:
                # Use .to_pandas() for mode() which can be more robust in cuDF
                mode_val = gdf[c].dropna().mode().to_pandas()
                fill_val = mode_val.iloc[0] if not mode_val.empty else "unknown"
                gdf[c] = gdf[c].fillna(fill_val)

            # 3. Feature Engineering (GPU-native)
            # Cardinality capping for categoricals
            for c in cat_cols:
                top_categories = gdf[c].value_counts().head(50).index
                gdf[c] = gdf[c].where(gdf[c].isin(top_categories), 'other')

            # Label encode categoricals
            for c in cat_cols:
                gdf[c] = gdf[c].astype('category').cat.codes.astype('int32')

            # 4. Scale numerics using cuML
            if num_cols and cuStandardScaler is not None:
                scaler = cuStandardScaler()
                # Explicitly convert to cupy array to avoid implicit host conversion warnings
                num_data = gdf[num_cols].to_cupy()
                scaled_data = scaler.fit_transform(num_data)
                gdf[num_cols] = cudf.DataFrame(scaled_data, columns=num_cols, index=gdf.index)

            # 5. Finalize target column
            gdf = self._binaryize_target(gdf)

            # 6. Save to efficient format
            out_path = os.path.join(self.out_dir, f"{dataset_name}_processed.parquet")
            gdf.to_parquet(out_path, compression="snappy")
            print(f"  💾 Saved: {out_path}  (rows={len(gdf):,}, cols={len(gdf.columns)})")
            
            return out_path

        except Exception as e:
            print(f"  🔥 ERROR processing {dataset_name} on GPU: {e}")
            return None
        finally:
            # IMPORTANT: Free GPU memory after processing each dataset
            cleanup_memory()


### 8. Sequential Processing and Memory-Safe Merging

This section implements the core orchestration logic. It demonstrates two key principles from your design:

1.  **Sequential Processing**: We iterate through each dataset, process it on the GPU using our `GPUMemoryEfficientDataProcessor`, and then clear the GPU memory. This prevents VRAM from being overwhelmed by loading all datasets at once.
2.  **Memory-Safe Merging**: After processing, we load the cleaned Parquet files one by one. A memory budget (`MEMORY_FRACTION_BUDGET`) is enforced. If the combined DataFrame exceeds this budget, it's adaptively downsampled to prevent crashing the Colab instance. This is a critical stability feature. We also apply a balanced sampling strategy to each dataset before merging to handle class imbalance early.

In [ ]:

def balanced_cap_sample(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    """
    Applies balanced sampling and a hard row cap to a single dataset's DataFrame.
    This is a key part of the 'Safe Memory Use' strategy.
    """
    fraud = df[df[target_col] == 1]
    legit = df[df[target_col] == 0]

    if len(fraud) == 0 or len(legit) == 0:
        # If no fraud or no legit, just cap the total rows
        return df.sample(n=min(len(df), MAX_ROWS_PER_DATASET), random_state=SEED, replace=len(df) < MAX_ROWS_PER_DATASET)

    # Keep all fraud, but cap legit rows based on the multiplier
    keep_legit_count = min(len(legit), MAJORITY_MULTIPLIER * len(fraud))
    
    df_balanced = pd.concat([
        fraud,
        legit.sample(n=keep_legit_count, random_state=SEED)
    ], ignore_index=True)

    # Apply the global per-dataset cap if the balanced set is still too large
    if len(df_balanced) > MAX_ROWS_PER_DATASET:
        df_balanced = df_balanced.sample(n=MAX_ROWS_PER_DATASET, random_state=SEED)
    
    print(f"  - Balanced sampling: {len(df)} rows -> {len(df_balanced)} rows ({len(fraud)} fraud, {len(df_balanced) - len(fraud)} legit)")
    return df_balanced

def load_all_processed_as_merged(parquets: List[str]) -> pd.DataFrame:
    """
    Loads processed parquet files into a single pandas DataFrame with a strict memory budget.
    If the budget is exceeded, the entire merged table is downsampled.
    """
    if not PSUTIL_AVAILABLE:
        print("⚠️ `psutil` not found. Cannot dynamically check memory. Relying on static cap.")
        total_ram = 12 * (1024**3) # Assume 12GB Colab RAM as a fallback
    else:
        total_ram = psutil.virtual_memory().total
    
    budget_bytes = int(total_ram * MEMORY_FRACTION_BUDGET)
    print(f"RAM Budget: {budget_bytes / 1e9:.2f} GB ({MEMORY_FRACTION_BUDGET * 100:.0f}% of total RAM)")

    frames = []
    dataset_ids = []
    current_id = 1
    total_rows = 0

    for path in parquets:
        if not (path and os.path.exists(path)):
            continue
        
        print(f"\\nLoading and sampling '{os.path.basename(path)}'...")
        df_processed = pd.read_parquet(path)
        
        # Apply balancing and row cap *before* merging
        df_sampled = balanced_cap_sample(df_processed, TARGET_COL)
        
        # Check against global row cap
        if total_rows + len(df_sampled) > GLOBAL_ROW_CAP:
            remaining_allowance = GLOBAL_ROW_CAP - total_rows
            if remaining_allowance <= 0:
                print("  - Global row cap reached. Skipping remaining datasets.")
                break
            df_sampled = df_sampled.sample(n=remaining_allowance, random_state=SEED)
            print(f"  - Capping to global limit: {len(df_sampled)} rows taken.")

        frames.append(df_sampled)
        dataset_ids.extend([current_id] * len(df_sampled))
        total_rows += len(df_sampled)
        current_id += 1
        
    if not frames:
        raise RuntimeError("No data was loaded. Check processing logs.")

    # Concatenate all frames at once
    print("\\nMerging all datasets...")
    df_merged = pd.concat(frames, ignore_index=True)
    df_merged['dataset_id'] = np.array(dataset_ids, dtype=np.int32)
    
    # Final memory check after merge
    final_mem_bytes = df_merged.memory_usage(deep=True).sum()
    print(f"Merged table memory: {final_mem_bytes / 1e9:.2f} GB")

    if final_mem_bytes > budget_bytes:
        downsample_frac = budget_bytes / final_mem_bytes
        keep_n = max(10_000, int(len(df_merged) * downsample_frac))
        print(f"  🔥 Memory budget exceeded! Downsampling to {keep_n:,} rows ({downsample_frac:.2%}).")
        df_merged = df_merged.sample(n=keep_n, random_state=SEED).reset_index(drop=True)
    
    cleanup_memory()
    return df_merged

def run_sequential_preprocessing():
    """
    Orchestrates the GPU processing of all enabled datasets, one by one.
    """
    if not GPU_AVAILABLE:
        print("🛑 This function requires a GPU and RAPIDS. Skipping.")
        return []
        
    processor = GPUMemoryEfficientDataProcessor(target_col=TARGET_COL, out_dir=OUT_DIR)
    processed_paths = []

    for dataset_config in DATASETS:
        if dataset_config.get("enabled", True):
            path = processor.process_dataset(dataset_config["path"], dataset_config["name"])
            if path:
                processed_paths.append(path)
    
    return processed_paths

# --- Execute the sequential processing and merging ---
# This flow is central to the memory-safe GPU architecture
processed_parquet_files = run_sequential_preprocessing()
if processed_parquet_files:
    df_unified = load_all_processed_as_merged(processed_parquet_files)
    print(f"\\n✅ Unified training table created: {df_unified.shape}")
    print(f"   - Fraud Rate: {df_unified[TARGET_COL].mean():.4f}")
    print(f"   - Datasets included: {df_unified['dataset_id'].nunique()}")
else:
    print("\\n🛑 No datasets were processed. The unified table could not be created.")
    # Create an empty dataframe to avoid errors in subsequent cells
    df_unified = pd.DataFrame()
